# 00. 데이터 준비 (AI Hub zip → Drive → Colab 로컬 압축 해제)

> ⚠️ **이 노트북은 CPU 런타임으로 실행하세요.** (런타임 → 런타임 유형 변경 → CPU)
> 압축 해제에는 GPU가 필요 없고, T4 사용량만 소모됩니다.

## 기본 방법: 웹에서 받아 Drive raw 폴더에 zip 그대로 업로드
1. AI Hub 웹사이트에서 세 데이터셋 파일을 내려받습니다.
2. **압축을 풀지 말고** zip 그대로 Drive의 아래 위치에 올립니다. (파일명이 다르면 설정 셀의 `UNZIP`을 맞추세요)
```
/content/drive/MyDrive/mmdl/data/raw/
  dialect_gs/  gs_wav.zip, gs_label.zip
  dialect_cc/  cc_wav.zip, cc_label.zip
  emotion/     emotion.zip, emotion.csv
```
3. 아래 압축 해제 셀을 실행하면 Colab 로컬(`/content/data`)에 풉니다. Drive에 직접 풀면 파일 수만 개 때문에 매우 느립니다.
   로컬 디스크는 **세션이 끝나면 지워지므로 세션마다 다시 실행**합니다. (02 전처리 노트북에도 같은 셀이 있습니다)

- AI Hub 데이터는 라이선스상 **저장소에 올리면 안 됩니다.** 원본은 Drive에만 둡니다.
- API로 직접 받고 싶으면 맨 아래 **(선택 사항) aihubshell** 섹션을 쓰세요.

In [ ]:
# [설정] 경로 — 여기만 수정하면 됩니다
RAW_DIR = "/content/drive/MyDrive/mmdl/data/raw"  # Drive에 올린 원본 zip (데이터셋별 하위 폴더)
LOCAL_DIR = "/content/data"                        # Colab 로컬 압축 해제 위치 (세션이 끝나면 지워짐)

# 원본 zip → 압축 해제 폴더
UNZIP = {
    f"{RAW_DIR}/dialect_gs/gs_wav.zip":   f"{LOCAL_DIR}/gs/wav",
    f"{RAW_DIR}/dialect_gs/gs_label.zip": f"{LOCAL_DIR}/gs/label",
    f"{RAW_DIR}/dialect_cc/cc_wav.zip":   f"{LOCAL_DIR}/cc/wav",
    f"{RAW_DIR}/dialect_cc/cc_label.zip": f"{LOCAL_DIR}/cc/label",
    f"{RAW_DIR}/emotion/emotion.zip":     f"{LOCAL_DIR}/emotion/wav",
}
EMO_CSV_SRC = f"{RAW_DIR}/emotion/emotion.csv"   # csv는 zip이 아니라 복사만
EMO_CSV = f"{LOCAL_DIR}/emotion/emotion.csv"

# (선택 사항) aihubshell로 받을 때만: datasetkey / filekey (AI Hub 데이터셋 페이지 또는 '파일 목록 조회' 셀로 확인)
# filekey는 쉼표로 여러 개 입력 가능 (예: "12345,12346"). 받은 파일은 RAW_DIR/<이름>/ 아래에 저장됨
DOWNLOADS = {
    "emotion":    {"datasetkey": "", "filekey": ""},  # 감정 분류를 위한 대화 음성 데이터셋 - 5차년도_2차
    "dialect_gs": {"datasetkey": "", "filekey": ""},  # 중·노년층 한국어 방언 - 강원·경상 (2인발화)
    "dialect_cc": {"datasetkey": "", "filekey": ""},  # 중·노년층 한국어 방언 - 충청·전라·제주 (2인발화)
}

In [ ]:
# Google Drive 마운트
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 원본 zip을 Colab 로컬에 압축 해제 (폴더 먼저 생성, 이미 풀린 폴더는 건너뜀)
import os, shutil, subprocess
from pathlib import Path
for zp, dest in UNZIP.items():
    done = Path(dest) / ".unzipped"  # 끝까지 풀렸을 때만 남기는 표시 (중간에 끊기면 다시 풂)
    if done.exists():
        print("[skip]", dest)
        continue
    os.makedirs(dest, exist_ok=True)
    print(f"{zp} → {dest}")
    r = subprocess.run(["unzip", "-q", "-o", "-O", "cp949", zp, "-d", dest])
    assert r.returncode <= 1, f"unzip 실패 (exit {r.returncode}): {zp}"  # 1 = 경고만 있음
    done.touch()
os.makedirs(os.path.dirname(EMO_CSV), exist_ok=True)
if not os.path.exists(EMO_CSV):
    shutil.copy(EMO_CSV_SRC, EMO_CSV)
print("압축 해제 완료:", LOCAL_DIR)

In [ ]:
# 결과 확인: 폴더별 확장자 개수와 용량
from collections import Counter
for d in sorted(set(UNZIP.values())):
    print(d, dict(Counter(p.suffix.lower() for p in Path(d).rglob("*") if p.is_file())))
!du -sh {LOCAL_DIR}/*/* 2>/dev/null

## (선택 사항) aihubshell로 직접 다운로드
웹 다운로드 대신 AI Hub API로 받을 때만 실행합니다. 설정 셀의 `DOWNLOADS`를 채운 뒤 아래 셀을 순서대로 실행하고, 받은 파일명이 `UNZIP`과 맞는지 확인한 다음 **위 압축 해제 셀을 다시 실행**하세요.

In [ ]:
# (선택) AI Hub API 다운로드 도구(aihubshell) 설치
!curl -s -o /usr/local/bin/aihubshell https://api.aihub.or.kr/api/aihubshell.do && chmod +x /usr/local/bin/aihubshell
!ls -l /usr/local/bin/aihubshell

In [ ]:
# (선택) datasetkey별 파일 목록 조회 (filekey 확인용, API 키 불필요)
import subprocess
for name, d in DOWNLOADS.items():
    if d["datasetkey"]:
        print(f"===== {name} (datasetkey={d['datasetkey']}) =====")
        print(subprocess.run(["aihubshell", "-mode", "l", "-datasetkey", d["datasetkey"]],
                             capture_output=True, text=True).stdout)
    else:
        print(f"[skip] {name}: datasetkey 비어 있음")

In [ ]:
# (선택) AI Hub API 키 입력 (getpass → 노트북 파일·출력에 남지 않음)
from getpass import getpass
AIHUB_API_KEY = getpass("AI Hub API 키: ").strip()
print("입력됨" if AIHUB_API_KEY else "비어 있음")

In [ ]:
# (선택) 데이터셋별 다운로드 → RAW_DIR/<이름>/ (출력에서 API 키는 *** 로 가림)
import subprocess
from pathlib import Path
assert AIHUB_API_KEY, "API 키를 먼저 입력하세요"
for name, d in DOWNLOADS.items():
    if not (d["datasetkey"] and d["filekey"]):
        print(f"[skip] {name}: datasetkey/filekey 비어 있음")
        continue
    out = Path(RAW_DIR) / name
    out.mkdir(parents=True, exist_ok=True)
    cmd = ["aihubshell", "-mode", "d", "-datasetkey", d["datasetkey"],
           "-filekey", d["filekey"], "-aihubapikey", AIHUB_API_KEY]
    print(f"===== {name} → {out} =====")
    p = subprocess.Popen(cmd, cwd=out, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line.replace(AIHUB_API_KEY, "***"), end="")
    print(f"{name}: {'완료' if p.wait() == 0 else f'실패 (exit {p.returncode})'}")

---
### ⚠️ 커밋 전에 출력 지우기
셀 출력에 AI Hub 전사 원문(방언·표준어·STT·발화문)이 남습니다. 라이선스상 재배포 금지이므로 저장소에 커밋하기 전에 **수정 → 모든 출력 지우기** 후 저장하세요.